<a href="https://colab.research.google.com/github/gianluca379/labo3-2026ba/blob/main/z101_EDA_01_con_seleccion_productos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json


In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp "/content/buckets/b1/kaggle/kaggle (2).json"  ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"


# 1 Exploratory Data Analysis

In [3]:
import os as os
import duckdb
import plotly.express as px

In [4]:
# defino los parametros
PARAM = {'experimento':'EDA-101',
  'kaggle_competition':'labo-iii-2026-ba'
}

In [5]:
# creo la carpeta del experimento
ruta = "/content/buckets/b1/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/EDA-101


## 1.1 Creacion de tablas

Para el Exploratory Data Analysis utilizo DuckDB
<br>Cargo los archivos a utilizar en TABLAS de DuckDB

In [6]:
con = duckdb.connect()

In [7]:
# creo la tabla del sell-in  transformando el campo periodo a tipo DATE

con.execute("""
    CREATE OR REPLACE TABLE tb_sellin AS
    SELECT CAST(strptime(CAST( periodo AS VARCHAR), '%Y%m') AS DATE) periodo,
           customer_id,
           product_id,
           plan_precios_cuidados,
           cust_request_qty,
           cust_request_tn,
           tn
    FROM read_csv_auto('/content/datasets/sell-in.txt.gz')
    ORDER BY product_id, periodo, customer_id
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
# creo la tabla que posee la definicion de los productos

con.execute("""
    CREATE OR REPLACE TABLE tb_productos AS
    SELECT *
    FROM read_csv_auto('/content/datasets/tb_productos.txt')
    ORDER BY product_id
""")

## 1.2 EDA Ventas Totales

In [9]:
# creo las ventas por producto

con.execute("""
    CREATE OR REPLACE TABLE tb_ventas_global AS
    SELECT periodo,
           SUM(tn) tn
    FROM  tb_sellin
    GROUP BY periodo
    ORDER BY periodo
""")



In [10]:
con.sql("""
SELECT  year(periodo) yean,
        round( SUM(tn) ) tn
FROM    tb_ventas_global
GROUP BY year(periodo)
ORDER BY year(periodo)
""").show()

┌───────┬──────────┐
│ yean  │    tn    │
│ int64 │  double  │
├───────┼──────────┤
│  2017 │ 500310.0 │
│  2018 │ 434448.0 │
│  2019 │ 390230.0 │
└───────┴──────────┘



¿Cómo interpreta lo que viene sucediendo año a año con las ventas?

In [11]:
# Grafico global de ventas
gra = px.line(
    con.sql("SELECT * FROM tb_ventas_global").df(),
    x="periodo",
    y="tn",
    title="Ventas Totales",
    labels={"periodo": "Periodo", "tn": "Toneladas"}
)

# Display the plot
gra.update_yaxes(rangemode="tozero")
gra.show()

In [12]:
# Grafico global de ventas con tendencia
gra = px.scatter(
    con.sql("SELECT * FROM tb_ventas_global").df(),
    x="periodo",
    y="tn",
    title="Ventas Totales con tendencia",
    labels={"periodo": "Periodo", "tn": "Toneladas"},
    trendline='ols',
    trendline_color_override='red'
)

# Display the plot
gra.update_traces(mode='lines')
gra.update_yaxes(rangemode="tozero")
gra.show()

¿Cómo es la tendencia de las ventas en los últimos 3 años?

## 1.3  EDA Clientes
¿Cuánto market share tiene los clientes?

In [13]:
# Creacion de tabla de clientes de 2019
con.execute("""
CREATE OR REPLACE TABLE tb_clientes_2019 AS
SELECT  customer_id,
        SUM(tn) tn,
        (SUM(tn) * 100.0) / SUM(SUM(tn)) OVER () tn_pct
FROM    tb_sellin
WHERE   periodo between '2019-01-01' AND '2019-12-01'
GROUP BY customer_id
ORDER BY tn DESC
""")

In [14]:
gra = px.bar(
    con.sql("SELECT tn_pct FROM tb_clientes_2019").df(),
    y="tn_pct",
    title="Densidad de tn por cliente",
    labels={ "tn_pct": "Toneladas Porcentaje"}
)

# Display the plot
gra.update_yaxes(rangemode="tozero")
gra.show()

In [15]:
# Cantidad de clientes distintos en 2019

con.sql("""
SELECT  COUNT( DISTINCT customer_id )
FROM    tb_clientes_2019
""")

┌─────────────────────────────┐
│ count(DISTINCT customer_id) │
│            int64            │
├─────────────────────────────┤
│                         534 │
└─────────────────────────────┘

In [16]:
# el market share acumulado de los mejores clientes
con.sql("""
SELECT  row_number() OVER(),
        customer_id,
        tn,
        tn_pct,
        SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado
FROM    tb_clientes_2019
ORDER BY tn DESC
LIMIT 20
""").show()

┌──────────────────────┬─────────────┬────────────────────┬────────────────────┬────────────────────┐
│ row_number() OVER () │ customer_id │         tn         │       tn_pct       │  tn_pct_acumulado  │
│        int64         │    int64    │       double       │       double       │       double       │
├──────────────────────┼─────────────┼────────────────────┼────────────────────┼────────────────────┤
│                    1 │       10001 │  33685.89001000001 │  8.632321989596619 │  8.632321989596619 │
│                    2 │       10002 │ 25948.003750000018 │  6.649416811928272 │ 15.281738801524892 │
│                    3 │       10003 │ 20514.406950000062 │  5.257007196943561 │ 20.538745998468453 │
│                    4 │       10004 │ 15890.077300000034 │   4.07197980081454 │ 24.610725799282992 │
│                    5 │       10005 │ 14958.134390000007 │ 3.8331607797747607 │ 28.443886579057754 │
│                    6 │       10006 │ 14147.636790000008 │ 3.6254632466854373 │  

Los primeros 13 productos representan el 50% de las ventas !

In [17]:
# el market share acumulado de los PEORES clientes
con.sql("""
SELECT  row_number() OVER(),
        customer_id,
        tn,
        tn_pct,
        SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado
FROM    tb_clientes_2019
ORDER BY tn ASC
LIMIT 300
""").show()

┌──────────────────────┬─────────────┬─────────────────────┬────────────────────────┬───────────────────┐
│ row_number() OVER () │ customer_id │         tn          │         tn_pct         │ tn_pct_acumulado  │
│        int64         │    int64    │       double        │         double         │      double       │
├──────────────────────┼─────────────┼─────────────────────┼────────────────────────┼───────────────────┤
│                  534 │       10604 │ 0.19665999999999995 │  5.039595041039766e-05 │ 99.99999999999983 │
│                  533 │       10524 │ 0.22125999999999998 │ 5.6699928749133474e-05 │ 99.99994960404942 │
│                  532 │       10618 │ 0.42128999999999994 │ 0.00010795947294008152 │ 99.99989290412067 │
│                  531 │       10574 │ 0.44403000000000004 │ 0.00011378680901418125 │ 99.99978494464773 │
│                  530 │       10581 │             0.50101 │ 0.00012838846290609855 │ 99.99967115783872 │
│                  529 │       10593 │  0.6977

Los 300 clientes menos importantes representan MENOS del 3% de las toneladas vendidas

In [18]:
# evolucion de algunos clientes

clientes=[10001, 10002, 10012]

gra=px.line(
    con.sql(f"SELECT customer_id, periodo, SUM(tn) tn FROM tb_sellin WHERE customer_id in {clientes} GROUP BY customer_id, periodo ORDER BY 1,2").df(),
    x="periodo",
    y="tn",
    color='customer_id',
    title='customer_id =' + str(clientes),
    labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

# Display the plot
gra.update_traces(mode='lines')
gra.update_yaxes(rangemode="tozero")
gra.show()

## 1.4 EDA productos individuales
Market shared de los productos

In [19]:
con.execute("""
CREATE OR REPLACE TABLE tb_productos_201912 AS
SELECT  product_id,
        SUM(tn) tn,
        (SUM(tn) * 100.0) / SUM(SUM(tn)) OVER () tn_pct
FROM    tb_sellin
WHERE   periodo between '2019-01-01' and '2019-12-01'
GROUP BY product_id
ORDER BY tn DESC
""")

In [20]:
con.sql("""
SELECT *
FROM   tb_productos_201912
""").show()

┌────────────┬──────────────────────┬────────────────────────┐
│ product_id │          tn          │         tn_pct         │
│   int64    │        double        │         double         │
├────────────┼──────────────────────┼────────────────────────┤
│      20001 │    17456.79264000003 │      4.473465149039151 │
│      20002 │   14105.245700000058 │     3.6146001363962186 │
│      20003 │    9419.716889999925 │      2.413889887462744 │
│      20005 │    8019.241249999978 │     2.0550050054104325 │
│      20004 │    7526.583939999965 │     1.9287569968470204 │
│      20009 │    6495.871040000001 │     1.6646272490805425 │
│      20032 │    6493.670259999988 │     1.6640632787777618 │
│      20006 │    5743.364500000031 │     1.4717904633928656 │
│      20007 │     5209.65367000002 │      1.335022109268118 │
│      20010 │    5154.895930000037 │     1.3209899301283614 │
│        ·   │            ·         │              ·         │
│        ·   │            ·         │              ·   

In [21]:
con.sql("""
SELECT  product_id,
        tn,
        tn_pct,
        SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado
FROM    tb_productos_201912
ORDER BY tn DESC
""").show()

┌────────────┬──────────────────────┬────────────────────────┬────────────────────┐
│ product_id │          tn          │         tn_pct         │  tn_pct_acumulado  │
│   int64    │        double        │         double         │       double       │
├────────────┼──────────────────────┼────────────────────────┼────────────────────┤
│      20001 │    17456.79264000003 │      4.473465149039151 │  4.473465149039151 │
│      20002 │   14105.245700000058 │     3.6146001363962186 │   8.08806528543537 │
│      20003 │    9419.716889999925 │      2.413889887462744 │ 10.501955172898114 │
│      20005 │    8019.241249999978 │     2.0550050054104325 │ 12.556960178308547 │
│      20004 │    7526.583939999965 │     1.9287569968470204 │ 14.485717175155568 │
│      20009 │    6495.871040000001 │     1.6646272490805425 │  16.15034442423611 │
│      20032 │    6493.670259999988 │     1.6640632787777618 │  17.81440770301387 │
│      20006 │    5743.364500000031 │     1.4717904633928656 │ 19.2861981664

In [22]:
def graficar_un_producto(producto):
  gra = px.scatter(
      con.sql(f"SELECT periodo, SUM(tn) tn FROM tb_sellin WHERE product_id={producto} GROUP BY periodo").df(),
      x="periodo",
      y="tn",
      title=con.sql(f"SELECT CONCAT_WS(', ', *COLUMNS('.*')) FROM tb_productos WHERE product_id={producto}").fetchone()[0],
      labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

  # Display the plot
  gra.update_traces(mode='lines')
  gra.update_yaxes(rangemode="tozero")
  gra.update_xaxes(range=['2017-01-01', '2019-12-01'])
  gra.show()

In [23]:
def graficar_productos(productos):
    for prod in productos:
      graficar_un_producto(prod)

In [24]:
# gnero graficos INDEPENDIENTES de productos

graficar_productos( [20001, 20002, 20003, 20004])

## 1.5  EDA multiples productos

In [25]:
productos = [ 20001, 20002]

tbl=con.sql(f"SELECT product_id, periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {productos} GROUP BY product_id, periodo ORDER BY 1, 2").df()
display(tbl)

,product_id,periodo,tn
0,20001,2017-01-01,934.77222
1,20001,2017-02-01,798.01620
2,20001,2017-03-01,1303.35771
3,20001,2017-04-01,1069.96130
4,20001,2017-05-01,1502.20132
...,...,...,...
67,20002,2019-08-01,813.78215
68,20002,2019-09-01,1090.18771
69,20002,2019-10-01,1979.53635
70,20002,2019-11-01,1423.57739


In [26]:
def graficar_multiples_productos(productos):
  gra = px.line(
      con.sql(f"SELECT product_id, periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {productos} GROUP BY product_id, periodo ORDER BY 1,2").df(),
      x="periodo",
      y="tn",
      color="product_id",
      title="Multiples Productos",
      labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

  # Display the plot
  gra.update_yaxes(rangemode="tozero")
  gra.update_xaxes(range=['2017-01-01', '2019-12-01'])
  gra.show()

In [27]:
def graficar_union_productos(productos):
  gra = px.line(
      con.sql(f"SELECT periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {productos} GROUP BY periodo ORDER BY 1").df(),
      x="periodo",
      y="tn",
      title="Productos in " + str(productos),
      labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

  # Display the plot
  gra.update_yaxes(rangemode="tozero")
  gra.update_xaxes(range=['2017-01-01', '2019-12-01'])
  gra.show()

In [28]:
graficar_multiples_productos( [20001, 20002, 20003] )

In [29]:
graficar_union_productos( [20001, 20002, 20003] )

## 1.6  EDA Estacionalidad Mayonesas

In [30]:
con.sql("SELECT * FROM tb_productos WHERE cat3='Mayonesa'")

┌─────────┬──────────┬──────────┬─────────┬──────────┬────────────┬──────────────────────┐
│  cat1   │   cat2   │   cat3   │  brand  │ sku_size │ product_id │     descripcion      │
│ varchar │ varchar  │ varchar  │ varchar │  int64   │   int64    │       varchar        │
├─────────┼──────────┼──────────┼─────────┼──────────┼────────────┼──────────────────────┤
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      475 │      20003 │ Regular sin TACC     │
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      240 │      20004 │ Regular sin TACC     │
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      120 │      20005 │ Regular sin TACC     │
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      950 │      20019 │ Regular sin TACC     │
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      475 │      20046 │ Light sin TACC       │
│ FOODS   │ ADEREZOS │ Mayonesa │ MAYOS3  │      475 │      20084 │ Reguar sin TACC      │
│ FOODS   │ ADEREZOS │ Mayonesa │ MAJESTA │      475 │      20107 │ Mayonesa Tradicional │

In [31]:
algunas_mayonesas = [20003, 20004, 20005]

In [32]:
con.sql(f"SELECT * FROM tb_productos WHERE product_id in {productos}")

┌─────────┬─────────────┬─────────┬─────────┬──────────┬────────────┬────────────────────┐
│  cat1   │    cat2     │  cat3   │  brand  │ sku_size │ product_id │    descripcion     │
│ varchar │   varchar   │ varchar │ varchar │  int64   │   int64    │      varchar       │
├─────────┼─────────────┼─────────┼─────────┼──────────┼────────────┼────────────────────┤
│ HC      │ ROPA LAVADO │ Liquido │ ARIEL   │     3000 │      20001 │ genoma             │
│ HC      │ ROPA LAVADO │ Liquido │ LIMPIEX │     3000 │      20002 │ Maquina 1er lavado │
└─────────┴─────────────┴─────────┴─────────┴──────────┴────────────┴────────────────────┘

In [33]:
graficar_multiples_productos(algunas_mayonesas)

In [34]:
graficar_union_productos(algunas_mayonesas)

¿Qué estacionalidad se observa para las mayonesas?

## 1.7 EDA Estacionalidad Sopas
Dado que entre que usualmente hay DOS MESES entre que el caminón sale por el portón de la planta de La Multinacional y es escaneado en la linea de caja del supermercado
<br> ¿En qué mes las familias empiezan a compar mas sopas?
<br> ¿En qué mes la multinacional vende más sopas?

In [35]:
con.sql("SELECT * FROM tb_productos WHERE cat3='Sopas'")

┌─────────┬────────────────┬─────────┬─────────┬──────────┬────────────┬─────────────────────────┐
│  cat1   │      cat2      │  cat3   │  brand  │ sku_size │ product_id │       descripcion       │
│ varchar │    varchar     │ varchar │ varchar │  int64   │   int64    │         varchar         │
├─────────┼────────────────┼─────────┼─────────┼──────────┼────────────┼─────────────────────────┤
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20234 │ Sopa 21                 │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20265 │ Sopa vegetales          │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20302 │ Sopa fideos y vegetales │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20305 │ Sopa fideos             │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20350 │ Sopla 22                │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20398 │ Sopa 1                  │
│ FOODS   

In [36]:
algunas_sopas=[20234, 20265, 20302]

In [37]:
graficar_multiples_productos(algunas_sopas)

In [38]:
graficar_union_productos(algunas_sopas)

## 1.8 EDA Productos Infantiles
¿Qué forma tiene el sell-out de los productos nuevos?
<br>¿Los productos nuevos, canibalizan a otros productos de la misma familia de productos de La Multinacional?

In [39]:
mostazas_infantes=[21144, 21146, 21154]

In [40]:
graficar_multiples_productos(mostazas_infantes)

In [41]:
mostazas_varias=[21144, 21146, 21154, 20884]

In [42]:
con.sql(f"SELECT * FROM tb_productos WHERE product_id in {mostazas_varias}")

┌─────────┬──────────┬─────────┬──────────┬──────────┬────────────┬──────────────────────────┐
│  cat1   │   cat2   │  cat3   │  brand   │ sku_size │ product_id │       descripcion        │
│ varchar │ varchar  │ varchar │ varchar  │  int64   │   int64    │         varchar          │
├─────────┼──────────┼─────────┼──────────┼──────────┼────────────┼──────────────────────────┤
│ FOODS   │ ADEREZOS │ Mostaza │ MOSTAZA1 │      275 │      20884 │ abejas                   │
│ FOODS   │ ADEREZOS │ Mostaza │ MOSTAZA1 │      180 │      21144 │ Pimienta                 │
│ FOODS   │ ADEREZOS │ Mostaza │ MOSTAZA1 │      180 │      21146 │ colchon de finas hierbas │
│ FOODS   │ ADEREZOS │ Mostaza │ MOSTAZA1 │      180 │      21154 │ Mostaza Ahumada          │
└─────────┴──────────┴─────────┴──────────┴──────────┴────────────┴──────────────────────────┘

In [43]:
graficar_multiples_productos(mostazas_varias)

Podria pasar que para 2019-09 el product_id=20884 se vendió menos porque fue canibalizado por los nuevos tres productos?

## 1.9 EDA Productos discontinuados

¿Los productos discontinuados son reemplazados por otros que toman su lugar?

In [44]:
mayonesas_discontinuadas=[20494, 20554]

In [45]:
graficar_multiples_productos(mayonesas_discontinuadas)

In [46]:
mayonesas=[20494, 20554, 20580]

In [47]:
graficar_multiples_productos(mayonesas)

# 2. Selección de productos por volumen para la búsqueda bayesiana

Reutiliza la conexión `con` y la tabla `tb_sellin` ya creadas más arriba en este mismo notebook.

**Ajustes respecto al análisis de concentración de clientes de la sección 1.3**:
1. Restringido a los 780 productos de `product_id_apredecir201912` (los únicos que importan para el score de Kaggle).
2. Volumen medido sobre 2019 (últimos 12 meses), no toda la historia 2017-2019 → evita que productos discontinuados (como los de la sección 1.9) distorsionen la selección.
3. Curva de cobertura completa para ELEGIR el umbral de corte en vez de adivinarlo, + muestra mixta con cola larga.

In [48]:
import random
import polars as pl

SEMILLA = 102191

## 2.1 Cargar la lista de productos a predecir (si no está ya en duckdb)

In [49]:
con.execute("""
    CREATE OR REPLACE TABLE tb_apredecir AS
    SELECT *
    FROM read_csv_auto('/content/datasets/product_id_apredecir201912.txt')
""")

## 2.2 Ventas 2019 solo de los productos a predecir

In [50]:
con.execute("""
    CREATE OR REPLACE TABLE tb_productos_apredecir_2019 AS
    SELECT  s.product_id,
            SUM(s.tn) AS tn,
            (SUM(s.tn) * 100.0) / SUM(SUM(s.tn)) OVER () AS tn_pct
    FROM    tb_sellin s
    INNER JOIN tb_apredecir a ON a.product_id = s.product_id
    WHERE   s.periodo BETWEEN '2019-01-01' AND '2019-12-01'
    GROUP BY s.product_id
    ORDER BY tn DESC
""")

ranking = con.sql("""
    SELECT  product_id,
            tn,
            tn_pct,
            SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado,
            ROW_NUMBER() OVER (ORDER BY tn_pct DESC) AS ranking_posicion
    FROM    tb_productos_apredecir_2019
    ORDER BY tn DESC
""").df()

total_productos = len(ranking)
print(f"Total de productos a predecir con ventas en 2019: {total_productos}")

Total de productos a predecir con ventas en 2019: 780


## 2.3 Curva de cobertura: cuántos productos necesito para cada % de ventas

Corré esta celda primero y mirá la curva **antes** de fijar el umbral en 2.4. Buscá visualmente el "codo": el punto donde agregar más productos ya no suma casi nada de cobertura.

In [51]:
print("--- Curva de cobertura ---")
for cobertura in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    n = int((ranking["tn_pct_acumulado"] <= cobertura * 100).sum())
    n = max(n, 1)
    print(f"Para cubrir el {cobertura*100:.0f}% de las ventas 2019, "
          f"necesitas {n} de {total_productos} productos "
          f"({n/total_productos*100:.1f}% del catalogo a predecir)")

# grafico de la curva, mismo estilo plotly que el resto del EDA
gra = px.line(
    ranking, x="ranking_posicion", y="tn_pct_acumulado",
    title="Cobertura acumulada de ventas 2019 (solo productos a predecir)",
    labels={"ranking_posicion": "Cantidad de productos (ordenados por volumen, desc)",
            "tn_pct_acumulado": "% acumulado de ventas"}
)
gra.add_hline(y=80, line_dash="dash", line_color="red",
              annotation_text="80%", annotation_position="bottom right")
gra.update_yaxes(rangemode="tozero")
gra.show()

--- Curva de cobertura ---
Para cubrir el 50% de las ventas 2019, necesitas 40 de 780 productos (5.1% del catalogo a predecir)
Para cubrir el 60% de las ventas 2019, necesitas 62 de 780 productos (7.9% del catalogo a predecir)
Para cubrir el 70% de las ventas 2019, necesitas 93 de 780 productos (11.9% del catalogo a predecir)
Para cubrir el 80% de las ventas 2019, necesitas 140 de 780 productos (17.9% del catalogo a predecir)
Para cubrir el 90% de las ventas 2019, necesitas 250 de 780 productos (32.1% del catalogo a predecir)
Para cubrir el 95% de las ventas 2019, necesitas 352 de 780 productos (45.1% del catalogo a predecir)


## 2.4 Selección: productos que cubren X% del volumen + cola larga al azar

In [52]:
def seleccionar_productos_por_cobertura(ranking_df, cobertura: float = 0.8) -> list:
    """
    Devuelve los product_id necesarios para cubrir 'cobertura' (ej 0.8=80%)
    del volumen 2019, ordenados de mayor a menor venta.
    """
    umbral = cobertura * 100
    seleccionados = ranking_df[ranking_df["tn_pct_acumulado"] <= umbral]["product_id"].tolist()
    if len(seleccionados) < len(ranking_df):
        siguiente = int(ranking_df.iloc[len(seleccionados)]["product_id"])
        seleccionados.append(siguiente)
    return seleccionados


def seleccionar_muestra_mixta(ranking_df, cobertura: float = 0.8,
                                n_cola_larga: int = 20, seed: int = SEMILLA) -> list:
    """
    Combina los productos "grandes" (cobertura elegida) + una muestra al
    azar de la cola larga, para que el tuning no quede ciego a productos
    chicos/intermitentes (justo los que mas fallan con estacionalidad).
    """
    grandes = seleccionar_productos_por_cobertura(ranking_df, cobertura)
    resto = ranking_df[~ranking_df["product_id"].isin(grandes)]
    random.seed(seed)
    resto_ids = resto["product_id"].tolist()
    cola_larga = random.sample(resto_ids, min(n_cola_larga, len(resto_ids)))
    return grandes + cola_larga

Ajustá estos dos parámetros mirando la curva de 2.3:

In [53]:
# --- PARAMETROS A DECIDIR VOS, mirando la curva de la seccion 2.3 ---
COBERTURA_ELEGIDA = 0.80   # ajustar segun donde veas el "codo" de la curva
N_COLA_LARGA = 20          # cuantos productos chicos sumar igual

productos_muestra = seleccionar_muestra_mixta(ranking, COBERTURA_ELEGIDA, N_COLA_LARGA)

print(f"Muestra final para la busqueda bayesiana: {len(productos_muestra)} productos "
      f"de {total_productos} ({len(productos_muestra)/total_productos*100:.1f}% del catalogo a predecir)")
print(f"  - {len(productos_muestra) - N_COLA_LARGA} productos que cubren el "
      f"{COBERTURA_ELEGIDA*100:.0f}% del volumen 2019")
print(f"  - {N_COLA_LARGA} productos de cola larga (al azar)")

Muestra final para la busqueda bayesiana: 161 productos de 780 (20.6% del catalogo a predecir)
  - 141 productos que cubren el 80% del volumen 2019
  - 20 productos de cola larga (al azar)


## 2.5 Exportar la lista para usarla en el notebook de AutoARIMA (polars)

In [54]:
ruta_salida = "/content/buckets/b1/exp/AA01/productos_muestra_bayesiana.csv"
pl.DataFrame({"product_id": [int(p) for p in productos_muestra]}).write_csv(ruta_salida)
print(f"Lista guardada en: {ruta_salida}")

Lista guardada en: /content/buckets/b1/exp/AA01/productos_muestra_bayesiana.csv


## 2.6 Cómo usar la lista en el notebook de AutoARIMA (z251)

En el notebook de AutoARIMA, reemplazar la selección aleatoria de productos por esto:

```python
productos_muestra = (
    pl.read_csv("/content/buckets/b1/exp/AA01/productos_muestra_bayesiana.csv")
    .get_column("product_id")
    .to_list()
)

study = correr_busqueda_bayesiana(
    series=series,
    productos_muestra=productos_muestra,
    db_path=db_path,
    n_trials=10,
    timeout_seg=1800,
)
```

In [55]:
from google.colab import _message
import json

# Obtiene el notebook actual desde la sesión de Colab
nb = _message.blocking_request('get_ipynb', timeout_sec=30)['ipynb']

# Limpia la metadata de widgets que rompe el preview de GitHub
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']

# Sobreescribe el notebook actual con la versión limpia
import IPython
IPython.display.Javascript('''
google.colab.kernel.invokeFunction('notebook.save', [], {});
''')

print("Metadata de widgets eliminada. Ahora usá Ctrl+S o File > Save para guardar.")

Metadata de widgets eliminada. Ahora usá Ctrl+S o File > Save para guardar.
